<a href="https://colab.research.google.com/github/ImagingDataCommons/CloudSegmentator/blob/main/workflows/TotalSegmentator/Notebooks/Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## How to Generate a Datatable for Terra to Run the TotalSegmentatortwoVmWorkflowOnTerra

This notebook prepares a Terra data table compatible with the [TotalSegmentatortwoVmWorkflowOnTerra](https://dockstore.org/workflows/github.com/ImagingDataCommons/Cloud-Resources-Workflows/TotalSegmentatortwoVmWorkflowOnTerra:dev?tab=info) workflow.

Steps:
1. **Configure** your cohort filters and batching parameters in the config cell below.
2. **Filter** the IDC index for collections, modality, and geometry-passing series using `idc-index` (no GCP auth required).
3. **Batch** the results into groups of series, assigning CPU/RAM hints per batch.
4. **Export** a Terra-compatible TSV data table.

## 1. Configuration

Edit the values in this cell to control which data is selected and how batches are formed.

In [ ]:
# ---------------------------------------------------------------------------
# COHORT FILTERS
# ---------------------------------------------------------------------------

# Collections to include. Supports wildcards via SQL LIKE patterns.
# Examples:
#   '%cmb%'   -> all CMB collections
#   '%nlst%'  -> NLST
#   '%tcga%'  -> all TCGA collections
# Use a list of patterns - any match is included.
COLLECTION_PATTERNS = [
    '%cmb%',
    '%cptac%',
]

# Modalities to include. Set to None to include all modalities.
# Examples: ['CT'], ['CT', 'MR'], None
MODALITIES = ['CT']

# Require series to pass all volume geometry checks (recommended for
# TotalSegmentator). Set to False to skip this filter.
REQUIRE_GEOMETRY_PASS = True

# IDC data version to embed in the output table (informational).
# Will be verified against the installed idc-index at runtime.
EXPECTED_IDC_VERSION = 'v24'

# ---------------------------------------------------------------------------
# BATCHING PARAMETERS
# ---------------------------------------------------------------------------

# Metric used to size batches:
#   'slices' -> sum of instanceCount across all series in the batch
#   'voxels' -> sum of (instanceCount * Rows * Columns) across series
#               (requires Rows/Columns in the IDC index; falls back to
#               512x512 for any series where they are NULL)
BATCH_METRIC = 'voxels'

# Target total metric value per batch. Series are accumulated until adding
# the next one would exceed this value, at which point a new batch begins.
# A single series that exceeds the target on its own gets a dedicated batch.
#
# Defaults:  slices -> 10 series x 120 slices = 1200
#            voxels -> 10 series x 120 slices x 512 x 512 = 314572800
BATCH_TARGET = 10*120*512*512

# CPU/RAM for the radiomics (dicomSegAndSR) VM.
# Series with max instanceCount above HIGH_RADIOMICS_THRESHOLD get the larger size.
HIGH_RADIOMICS_THRESHOLD = 300
RADIOMICS_CPU_LARGE = 8;  RADIOMICS_RAM_LARGE = 32
RADIOMICS_CPU_SMALL = 4;  RADIOMICS_RAM_SMALL = 16

# CPU/RAM for the inference VM.
# MOOSE requires 16 GB RAM minimum; runs CPU-only (no GPU required).
HIGH_INFERENCE_THRESHOLD = 750
INFERENCE_CPU_LARGE = 8;  INFERENCE_RAM_LARGE = 52
INFERENCE_CPU_SMALL = 4;  INFERENCE_RAM_SMALL = 16


## 2. Install / verify idc-index

In [ ]:
import subprocess, sys

REQUIRED_IDC_INDEX_VERSION = '0.12.3'

try:
    import idc_index
    if idc_index.__version__ < REQUIRED_IDC_INDEX_VERSION:
        print(f'Upgrading idc-index {idc_index.__version__} -> {REQUIRED_IDC_INDEX_VERSION} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade',
                        '--quiet', f'idc-index=={REQUIRED_IDC_INDEX_VERSION}'], check=True)
        import importlib; importlib.reload(idc_index)
    else:
        print(f'idc-index {idc_index.__version__} OK')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                    f'idc-index=={REQUIRED_IDC_INDEX_VERSION}'], check=True)
    import idc_index

from idc_index import IDCClient
client = IDCClient()

idc_version = client.get_idc_version()
print(f'IDC data version: {idc_version}')
assert idc_version == EXPECTED_IDC_VERSION, (
    f"IDC version mismatch: got '{idc_version}', expected '{EXPECTED_IDC_VERSION}'. "
    "Update EXPECTED_IDC_VERSION in the config cell if intentional."
)

## 3. Discover matching collections

In [ ]:
pattern_clauses = ' OR '.join(
    f"lower(collection_id) LIKE '{p.lower()}'"
    for p in COLLECTION_PATTERNS
)

collections = client.sql_query(f"""
    SELECT DISTINCT collection_id
    FROM index
    WHERE {pattern_clauses}
    ORDER BY collection_id
""")

print(f'Matched {len(collections)} collections:')
for cid in collections['collection_id'].tolist():
    print(f'  {cid}')

## 4. Load geometry index and query series

In [ ]:
import pandas as pd

# Build WHERE clauses from config
collection_filter = f'({pattern_clauses})'

modality_filter = ''
if MODALITIES:
    mod_list = ', '.join(f"'{m}'" for m in MODALITIES)
    modality_filter = f'AND i.Modality IN ({mod_list})'

if REQUIRE_GEOMETRY_PASS:
    print('Fetching volume_geometry_index ...')
    client.fetch_index('volume_geometry_index')
    join_clause = 'INNER JOIN volume_geometry_index AS g USING (SeriesInstanceUID)'
    geometry_filter = 'AND g.regularly_spaced_3d_volume = TRUE'
    geometry_cols = ', g.obliquity_degrees, g.regularly_spaced_3d_volume'
else:
    join_clause = ''
    geometry_filter = ''
    geometry_cols = ''

# Fetch ct_index to get real Rows/Columns instead of assuming 512x512
print('Fetching ct_index ...')
client.fetch_index('ct_index')
ct_join = 'LEFT JOIN ct_index AS ct USING (SeriesInstanceUID)'
ct_cols = ', COALESCE(ct.Rows, 512) AS Rows, COALESCE(ct.Columns, 512) AS Columns'

query = f"""
    SELECT
        i.collection_id,
        i.PatientID,
        i.StudyInstanceUID,
        i.SeriesInstanceUID,
        i.Modality,
        i.SeriesDescription,
        i.instanceCount
        {geometry_cols}
        {ct_cols}
    FROM index AS i
    {join_clause}
    {ct_join}
    WHERE {collection_filter}
    {modality_filter}
    {geometry_filter}
    ORDER BY i.collection_id, i.PatientID, i.StudyInstanceUID
"""

print('Running query ...')
df = client.sql_query(query)
print(f'\nSeries matching all filters: {len(df)}')
df.head()


## 5. Per-collection summary

In [ ]:
summary = (
    df.groupby('collection_id')
    .agg(
        series=('SeriesInstanceUID', 'count'),
        patients=('PatientID', 'nunique'),
        studies=('StudyInstanceUID', 'nunique'),
        median_slices=('instanceCount', 'median'),
    )
    .reset_index()
    .sort_values('collection_id')
)
summary

## 6. Batch series and build Terra data table

In [ ]:
import json
from datetime import datetime

now = datetime.now().strftime('%Y_%m_%d_%H_%M')
batch_id_col = f'entity:twoVM_{now}_id'

def series_metric(row):
    count = int(row['instanceCount']) if row['instanceCount'] is not None else 0
    if BATCH_METRIC == 'voxels':
        rows = int(row.get('Rows', 512) or 512)
        cols = int(row.get('Columns', 512) or 512)
        return count * rows * cols
    return count

rows = []
batch_num = 1
temp_batch = []   # list of (SeriesInstanceUID, instanceCount)
temp_metric = 0   # running metric total for the current batch

def flush_batch(series_tuples):
    global batch_num
    uids = [s[0] for s in series_tuples]
    max_slices = max(s[1] for s in series_tuples)

    r_cpu = RADIOMICS_CPU_LARGE if max_slices > HIGH_RADIOMICS_THRESHOLD else RADIOMICS_CPU_SMALL
    r_ram = RADIOMICS_RAM_LARGE if max_slices > HIGH_RADIOMICS_THRESHOLD else RADIOMICS_RAM_SMALL
    i_cpu = INFERENCE_CPU_LARGE if max_slices > HIGH_INFERENCE_THRESHOLD else INFERENCE_CPU_SMALL
    i_ram = INFERENCE_RAM_LARGE if max_slices > HIGH_INFERENCE_THRESHOLD else INFERENCE_RAM_SMALL

    rows.append({
        batch_id_col:         batch_num,
        'SeriesInstanceUIDs': json.dumps({'SeriesInstanceUIDs': uids}),
        'idc-version':        idc_version,
        'dicomSegAndSRcpu':   r_cpu,
        'dicomSegAndSRram':   r_ram,
        'inferenceCpu':       i_cpu,
        'inferenceRam':       i_ram,
    })
    batch_num += 1

for _, row in df.iterrows():
    uid    = row['SeriesInstanceUID']
    count  = int(row['instanceCount']) if row['instanceCount'] is not None else 0
    metric = series_metric(row)

    # If this series would push the batch over the target, flush first
    # (unless the batch is empty, in which case this series gets its own batch)
    if temp_batch and temp_metric + metric > BATCH_TARGET:
        flush_batch(temp_batch)
        temp_batch = []
        temp_metric = 0

    temp_batch.append((uid, count))
    temp_metric += metric

if temp_batch:
    flush_batch(temp_batch)

batch_df = pd.DataFrame(rows)
print(f'Batching metric : {BATCH_METRIC}')
print(f'Target per batch: {BATCH_TARGET:,}')
print(f'Total Terra rows (batches): {len(batch_df)}')
batch_df.head()


In [ ]:
import ast

series_counts = batch_df['SeriesInstanceUIDs'].apply(
    lambda v: len(ast.literal_eval(v)['SeriesInstanceUIDs'])
)
batch_df.assign(series_count=series_counts)[
    [batch_id_col, 'series_count']
]


## 7. Export Terra TSV

In [ ]:
output_file = f'terra_data_table_{now}.tsv'
batch_df.to_csv(output_file, sep='\t', index=False)
print(f'Saved {len(batch_df)} rows -> {output_file}')